In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Load data

In [ ]:
import pandas as pd
df = pd.read_csv('data.csv')
df.head()

In [ ]:
df['Maker_Model']= df['Make']+ " " + df['Model']

In [ ]:
df

In [ ]:
# Select features from original dataset to form a new dataframe
df1 = df[['Engine Fuel Type','Transmission Type','Driven_Wheels','Market Category','Vehicle Size', 'Vehicle Style', 'Maker_Model']]
df1.head()

In [ ]:
# For each row, combine all the columns into one column
df2 = df1.apply(lambda x: ','.join(x.astype(str)), axis=1)
df2.head()

In [ ]:
df2.shape

In [ ]:
# Store them in a pandas dataframe
df_clean = pd.DataFrame({'clean': df2})
df_clean.head()

In [ ]:
# Create the list of list format of the custom corpus for gensim modeling
sent = [row.split(',') for row in df_clean['clean']]
# show the example of list of list format of the custom corpus for gensim modeling
sent[:2]

## Train Word2Vec model (skip-gram)

In [ ]:
# from gensim import models
from gensim.models import Word2Vec
model = Word2Vec(sent, min_count=1,vector_size= 300,workers=5, window=10, sg = 1, epochs=100)
# model = Word2Vec(sent, min_count=1,size= 300,workers=5, window=10, sg = 1, iter=100)

**vector_size**: The number of dimensions of the embeddings and the default is 100.

**window**: The maximum distance between a target word and words around the target word. The default window is 5.

**min_count**: The minimum count of words to consider when training the model; words with occurrence less than this count will be ignored. The default for min_count is 5.

**workers**: The number of partitions during training and the default workers is 3.

**sg**: The training algorithm, either CBOW(0) or skip gram(1). The default training algorithm is CBOW.

In [ ]:
model.wv['Toyota Camry']

In [ ]:
model.wv.similarity('Porsche 718 Cayman', 'Nissan Van')

In [ ]:
model.wv.similarity('Porsche 718 Cayman', 'Mercedes-Benz SLK-Class')

In [ ]:
model.wv.most_similar('Mercedes-Benz SLK-Class')[:5]

In [ ]:
model

In [ ]:
import numpy as np
def cosine_distance (model, word,target_list , num) :
    cosine_dict ={}
    word_list = []
    a = model.wv[word]
    for item in target_list :
        if item != word :
            b = model.wv[item]
            cos_sim = np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b))
            cosine_dict[item] = cos_sim
    dist_sort=sorted(cosine_dict.items(), key=lambda dist: dist[1],reverse = True) ## in Descending order
    for item in dist_sort:
        word_list.append((item[0], item[1]))
    return word_list[0:num]

In [ ]:
# only get the unique Maker_Model
Maker_Model = list(df.Maker_Model.unique())

# Show the most similar Mercedes-Benz SLK-Class by cosine distance
cosine_distance(model,'Mercedes-Benz SLK-Class',Maker_Model,5)


# Activity

1. Use the customer complaint data to estimate the word embedding vectors
2. Find similar words to 'debt', 'collection', 'risk'